In [ ]:
!git clone https://github.com/srmty09/smol-training-kit.git

In [ ]:
!pip install -r /kaggle/working/smol-training-kit/requirements.txt

In [ ]:
import sys
sys.path.append('/kaggle/working/smol-training-kit')

from pre_training_kit_lm import get_trainer, TrainingConfig

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Subset

In [ ]:
from datasets import load_dataset

ds = load_dataset("teknium/GPTeacher-General-Instruct")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
MAX_LENGTH = 350

def preprocess(example):
    instruction = example["instruction"].strip()
    input_text = example.get("input", "").strip()
    output_text = example["response"].strip()

    if input_text:
        prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    else:
        prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(output_text, add_special_tokens=False)["input_ids"]

    max_answer_len = MAX_LENGTH - len(prompt_ids) - 1

    if max_answer_len <= 0:
        return {"input_ids": [], "labels": [], "keep": False}

    answer_ids = answer_ids[:max_answer_len]
    answer_ids = answer_ids + [tokenizer.eos_token_id]

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids

    return {"input_ids": input_ids, "labels": labels, "keep": True}

In [ ]:
train_dataset = ds["train"].map(
    preprocess,
    remove_columns=ds["train"].column_names,
)

train_dataset = train_dataset.filter(lambda x: x["keep"])
train_dataset = train_dataset.remove_columns(["keep"])

In [ ]:
def sft_collate_fn(batch, tokenizer):
    input_ids_list = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    labels_list = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]

    lengths = torch.tensor([len(x) for x in input_ids_list], dtype=torch.long)

    input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = pad_sequence(labels_list, batch_first=True, padding_value=-100)

    max_len = input_ids.size(1)
    attention_mask = (torch.arange(max_len)[None, :] < lengths[:, None]).long()

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
import random

def make_length_grouped_dataset(dataset, bucket_size=1024, seed=42):
    dataset = dataset.map(lambda x: {"length": len(x["input_ids"])})
    dataset = dataset.sort("length")

    indices = list(range(len(dataset)))
    buckets = [indices[i:i + bucket_size] for i in range(0, len(indices), bucket_size)]

    rng = random.Random(seed)
    rng.shuffle(buckets)

    new_indices = []
    for bucket in buckets:
        rng.shuffle(bucket)
        new_indices.extend(bucket)

    return Subset(dataset, new_indices)

In [ ]:
cfg = TrainingConfig.from_file("/kaggle/input/datasets/smrtytest/ift360mconfig/ift360Mconfig.yaml")
cfg.show()

train_dataset_grouped = make_length_grouped_dataset(
    train_dataset,
    bucket_size=1024,
    seed=cfg.seed,
)

train_loader = DataLoader(
    train_dataset_grouped,
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=lambda batch: sft_collate_fn(batch, tokenizer),
)

In [ ]:
trainer = get_trainer(
    model=model,
    training_cfg=cfg,
    train_dataloader=train_loader,
    tokenizer=tokenizer,
)

trainer.train()